# Running multiple circuits with `CircuitBinding`

`CircuitBinding` groups circuits, parameter values, and measurements into related
runs. Providers can package these runs into one submission and, when compatible,
share hardware executions to reduce QPU time and cost. Providers without native
support execute the binding as a standard batch.

The three **axes** of a binding are its circuits, parameter value sets, and
measurements. Each logical execution selects one entry from each axis.

A binding lets you:

- Run several circuits together.
- Evaluate a circuit at different parameter values.
- Measure the same circuit with different observables.
- Combine these patterns using **PRODUCT**, **ZIP**, or nested bindings.

All executions in one binding must use the same job type. A binding cannot
mix state-vector, sample, and observable jobs.

This is MPQP's counterpart to Qiskit's PUBs and Braket's ProgramSets. The examples
below use local simulators.

In [1]:
# First we setup here the circuits, values and observables to be used
from mpqp import BindingMode, CircuitBinding
from mpqp.execution.devices import (
    IBMDevice,
    AWSDevice,
    QUANTINUUMDevice,
)

from mpqp import (
    CNOT,
    BasisMeasure,
    ExpectationMeasure,
    H,
    Observable,
    QCircuit,
    Ry,
    U,
    X,
    pI,
    pZ,
    pX,
    run,
)
from sympy import Symbol
import numpy as np

theta, phi, psi = Symbol('θ'), Symbol('phi'), Symbol('psi')
c1 = QCircuit([U(theta, phi, psi, 0)], label="c1")
c2 = QCircuit([H(0)], label="c2")
c3 = QCircuit([H(0), CNOT(0, 1)], label="c3")
c4 = QCircuit([H(0), H(1), CNOT(0, 1)], label="c4")

v1 = {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
v2 = {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
v3 = {'θ': 3.0, 'phi': 3.0, 'psi': 3.0}
v4 = {'θ': 4.0, 'phi': 4.0, 'psi': 4.0}
v_t0 = {'θ': 0}
v_t1 = {'θ': np.pi}

m1 = ExpectationMeasure(Observable(pI), label="Exp1", shots=2024)
m1_bis = ExpectationMeasure(Observable(pZ), label="Exp1bis", shots=2024)
m2 = ExpectationMeasure(Observable(pX @ pZ), label="Exp2", shots=2024)
m2_bis = ExpectationMeasure(Observable(pZ @ pZ), label="Exp2bis", shots=2024)
m3 = BasisMeasure(label="b3", shots=2024)
m3_bis = BasisMeasure(label="b3_bis", shots=1024)
m4 = None

m_I = ExpectationMeasure(Observable(pI), label="Exp_I", shots=2024)
m_Z = ExpectationMeasure(Observable(pZ), label="Exp_Z", shots=2024)

## 1. Run a list of circuits

For a basic batch, pass the circuits directly to `CircuitBinding`.

- **State vectors:** leave the circuits without measurements.
- **Samples:** include a `BasisMeasure` in each circuit.
- **Expectation values:** include an `ExpectationMeasure` in each circuit.

You can also supply measurements through the binding's `measurements` argument,
provided the circuits do not already contain measurements. The sampling example
below shows both forms. When a binding measurement has no explicit targets, MPQP
resolves its targets separately for the number of qubits in each circuit.

The same job-type rule applies to nested bindings. For example, an outer
`ExpectationMeasure` cannot be combined with an inner `BasisMeasure`.

In [2]:
# Example 1: Two state vector jobs
# Note: state vector jobs are not supported by braket's ProgramSets
binding = CircuitBinding([c3, c4])
print(
    run(
        binding,
        [
            IBMDevice.AER_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_STATEVECTOR_SIMULATOR,
        ],
    )
)

BatchResult: 4 results
    Result: c3, IBMDevice, AER_SIMULATOR
      State vector: [0.70711, 0, 0, 0.70711]
      Probabilities: [0.5, 0, 0, 0.5]
      Number of qubits: 2
    Result: c4, IBMDevice, AER_SIMULATOR
      State vector: [0.5, 0.5, 0.5, 0.5]
      Probabilities: [0.25, 0.25, 0.25, 0.25]
      Number of qubits: 2
    Result: c3, QUANTINUUMDevice, TKET_AER_STATEVECTOR_SIMULATOR
      State vector: [0.70711, 0, 0, 0.70711]
      Probabilities: [0.5, 0, 0, 0.5]
      Number of qubits: 2
    Result: c4, QUANTINUUMDevice, TKET_AER_STATEVECTOR_SIMULATOR
      State vector: [0.5, 0.5, 0.5, 0.5]
      Probabilities: [0.25, 0.25, 0.25, 0.25]
      Number of qubits: 2


In [3]:
binding_with_basismeasure = CircuitBinding([c3 + QCircuit([m3]), c4 + QCircuit([m3])])
# or you can do
binding_with_basismeasure2 = CircuitBinding(
    [c3, c4], measurements=[m3], mode=BindingMode.PRODUCT
)
print(
    run(
        binding_with_basismeasure,
        [
            AWSDevice.BRAKET_LOCAL_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_SIMULATOR,
        ],
    )
)

BatchResult: 4 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [1039, 0, 0, 985]
      Probabilities: [0.51334, 0, 0, 0.48666]
      Samples:
        State: 00, Index: 0, Count: 1039, Probability: 0.5133399
        State: 11, Index: 3, Count: 985, Probability: 0.4866601
      Error: None
    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [490, 491, 514, 529]
      Probabilities: [0.24209, 0.24259, 0.25395, 0.26136]
      Samples:
        State: 00, Index: 0, Count: 490, Probability: 0.2420949
        State: 01, Index: 1, Count: 491, Probability: 0.2425889
        State: 10, Index: 2, Count: 514, Probability: 0.2539526
        State: 11, Index: 3, Count: 529, Probability: 0.2613636
      Error: None
    Result: c3, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Counts: [991, 0, 0, 1033]
      Probabilities: [0.48962, 0, 0, 0.51038]
      Samples:
        State: 00, Index: 0, Count: 991, Probability: 0.4896245
        State: 11, Index: 3, Count: 1033, Pro

In [4]:
binding_with_observables = CircuitBinding([c3 + QCircuit([m2]), c4 + QCircuit([m2])])

print(
    run(
        binding_with_observables,
        [
            IBMDevice.AER_SIMULATOR,
            AWSDevice.BRAKET_LOCAL_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_SIMULATOR,
        ],
    )
)

BatchResult: 6 results
    Result: c3, IBMDevice, AER_SIMULATOR
      Expectation value: -0.011358024691358024
      Error/Variance: 0.02222078879015875
      Observables: [pX@pZ]
    Result: c4, IBMDevice, AER_SIMULATOR
      Expectation value: -0.0004938271604938272
      Error/Variance: 0.022222219512608006
      Observables: [pX@pZ]
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: 0.03359683794466403
      Error/Variance: 0.000493513464664091
      Observables: [pX@pZ]
    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: -0.0029644268774703555
      Error/Variance: 0.0004940668044334427
      Observables: [pX@pZ]
    Result: c3, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Expectation value: 0.0039525691699604515
      Error/Variance: 0.0004940634274688521
      Observables: [pX@pZ]
    Result: c4, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Expectation value: 0.0019762845849802257
      Error/Variance: 0.0004940692165510075
      Observ

## 2. Choose a binding mode

### PRODUCT: evaluate every combination

`BindingMode.PRODUCT` combines every circuit with every parameter set and every
measurement. It is the default mode.

For example:

```python
CircuitBinding(
    [c1, c2],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.PRODUCT,
)
```

This produces **8 logical executions**, in the following order:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c1` | `v1` | `m1_bis` |
| `c1` | `v2` | `m1` |
| `c1` | `v2` | `m1_bis` |
| `c2` | `v1` | `m1` |
| `c2` | `v1` | `m1_bis` |
| `c2` | `v2` | `m1` |
| `c2` | `v2` | `m1_bis` |

The runnable example below uses only `v1`, giving **4 logical executions**.
The number of logical executions need not equal the number of hardware executions.

In [5]:
binding_product = CircuitBinding(
    [c1, c2], values=[v1], measurements=[m1, m1_bis], mode=BindingMode.PRODUCT
)

print(
    run(
        binding_product,
        [
            AWSDevice.BRAKET_LOCAL_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_SIMULATOR,
        ],
    )
)

BatchResult: 8 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 0.5237154150197628
      Error/Variance: 0.0003585583814578447
      Observables: [pZ]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 0.0029644268774703555
      Error/Variance: 0.0004940668044334427
      Observables: [pZ]
    Result: c1, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]


### ZIP: pair corresponding entries

`BindingMode.ZIP` pairs entries from the three axes by position. Each axis must
have either **one entry** or **as many entries as the longest axis**. An axis with
one entry is broadcast: its entry is reused for every position.

```python
CircuitBinding(
    [c1, c2],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)
```

This produces **2 logical executions**:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c2` | `v2` | `m1_bis` |

For example, a circuit axis of length one is reused for every pair of values and
measurements:

```python
CircuitBinding(
    [c1],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)
```

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c1` | `v2` | `m1_bis` |

The same broadcasting rule applies to **all three axes**.

An example of failing inputs would be inputs of lengths `(2, 3, 1)`. They are 
incompatible: the circuit axis has neither one entry nor the maximum length, 
three. Invalid lengths raise `ValueError` when the binding is constructed. 

Lastly, as you will see bellow, an omitted values or measurements axis counts as
one unspecified entry.

The next two cells first pair individual circuits, then reuse a nested binding
containing both circuits for each pair.

In [6]:
binding_zip = CircuitBinding(
    [c1, c2], values=[v1, v2], measurements=[m1, m1_bis], mode=BindingMode.ZIP
)

print(
    run(
        binding_zip,
        [
            AWSDevice.BRAKET_LOCAL_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_SIMULATOR,
        ],
    )
)

BatchResult: 4 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: 0.009881422924901186
      Error/Variance: 0.0004940229038937644
      Observables: [pZ]
    Result: c1, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.015810276679841806
      Error/Variance: 0.0004939476458257446
      Observables: [pZ]


In [7]:
binding_zip_1_circuit = CircuitBinding(
    [CircuitBinding([c1, c2])],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)

print(
    run(
        binding_zip_1_circuit,
        [
            AWSDevice.BRAKET_LOCAL_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_SIMULATOR,
        ],
    )
)

BatchResult: 8 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.4209486166007905
      Error/Variance: 0.0004065228568092296
      Observables: [pZ]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.04940711462450593
      Error/Variance: 0.0004928650874626882
      Observables: [pZ]
    Result: c1, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]


**A nested binding counts as one entry in the outer circuit list.** In the
preceding example, `[CircuitBinding([c1, c2])]` is therefore reused for both
value/measurement pairs, applying each pair to both inner circuits.

The next cell shows the simpler case: reuse only `c1` for the two pairs.

In [8]:
binding_zip_1_cb = CircuitBinding([c1], [v1, v2], [m1, m1_bis], mode=BindingMode.ZIP)

print(
    run(
        binding_zip_1_cb,
        [
            AWSDevice.BRAKET_LOCAL_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_SIMULATOR,
        ],
    )
)

BatchResult: 4 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.39031620553359686
      Error/Variance: 0.00041880101763728015
      Observables: [pZ]
    Result: c1, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, QUANTINUUMDevice, TKET_AER_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.4209486166007905
      Error/Variance: 0.0004065228568092296
      Observables: [pZ]


## 3. Nest bindings

A `CircuitBinding` can contain other bindings. This lets you define an inner
parameter sweep, then supply measurements at the outer level.

### Supply a missing measurement

```python
inner = CircuitBinding([c1, c2], values=[v1, v2])
outer = CircuitBinding(
    [inner, c3],
    measurements=[m1, m2],
    mode=BindingMode.ZIP,
)
```

ZIP pairs `inner` with `m1` and `c3` with `m2`. The inner sweep expands to four
executions, all using `m1`:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c1` | `v2` | `m1` |
| `c2` | `v1` | `m1` |
| `c2` | `v2` | `m1` |
| `c3` | None | `m2` |

### Combine options from both levels

When parameters or measurements are supplied at both levels:

- **Disjoint parameter names** are merged into one mapping.
- **Overlapping parameter names** contribute alternative parameter sets.
- **Measurements at both levels** contribute alternative measurements.

The runnable example below uses `v1` inside and `v2` outside. These mappings
contain the same parameter names, so they remain alternatives. Combined with
`m1` inside and `m1_bis` outside, they produce:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v2` | `m1_bis` |
| `c1` | `v2` | `m1` |
| `c1` | `v1` | `m1_bis` |
| `c1` | `v1` | `m1` |
| `c2` | `v2` | `m1_bis` |

Use `binding.unroll()` to inspect the resolved execution sequence without
submitting any jobs.

In [9]:
binding_embedded = CircuitBinding(
    [CircuitBinding([c1], values=[v1], measurements=m1), c2],
    values=[v2],
    measurements=[m1_bis],
    mode=BindingMode.PRODUCT,
)

print("\n".join(str(c) for c in binding_embedded.unroll()) + "\n")
print(
    run(
        binding_embedded,
        [
            AWSDevice.BRAKET_LOCAL_SIMULATOR,
            QUANTINUUMDevice.TKET_AER_SIMULATOR,
        ],
    )
)

(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}, ExpectationMeasure(Observable(pZ, 'Exp1bis_0'), [0], shots=2024, label='Exp1bis'))
(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}, ExpectationMeasure(Observable(pI, 'Exp1_0'), [0], shots=2024, label='Exp1'))
(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}, ExpectationMeasure(Observable(pZ, 'Exp1bis_0'), [0], shots=2024, label='Exp1bis'))
(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}, ExpectationMeasure(Observable(pI, 'Exp1_0'), [0], shots=2024, label='Exp1'))
(QCircuit([H(0)], label="c2"), {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}, ExpectationMeasure(Observable(pZ, 'Exp1bis_0'), [0], shots=2024, label='Exp1bis'))

BatchResult: 10 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.40118577075098816
      Error/Variance: 0.0004145503840

## 4. Run bindings with Quantinuum

Quantinuum supports the three MPQP job types through `CircuitBinding`. The
selected device must support the requested result:

- Use a sampling simulator, such as `TKET_AER_SIMULATOR`, for finite-shot
  samples and sampled expectation values.
- Use a state-vector simulator, such as `TKET_AER_STATEVECTOR_SIMULATOR`,
  when the full quantum state is required.
- Exact expectation values with `shots=0` are supported by local TKET
  simulators. Nexus currently requires sampled observables with
  `optimize_measurement=True`.

### Bind parameters and observables

This example is close to a variational workflow: the same symbolic circuit is
evaluated for several parameter values and observables. For Quantinuum, MPQP
translates and compiles the symbolic TKET circuit once, then assigns each
parameter set for its execution. The original symbolic circuit remains unchanged
and can be reused with other values.

Parameter mappings belong in the binding's `values` argument. Once a binding is
created, do not pass another `values` argument to `run` or `submit`.

In [10]:
angle = Symbol("angle")
binding = CircuitBinding(
    QCircuit([Ry(angle, 0)], label="parameterized"),
    values=[{angle: 0.0}, {angle: np.pi / 2}],
    measurements=ExpectationMeasure(
        [Observable(pZ, label="Z"), Observable(pX, label="X")]
    ),
)

batch_result = run(
    binding,
    QUANTINUUMDevice.TKET_AER_SIMULATOR,
)
print("Parameterized observable results:")
for result in batch_result.results:
    print(result.job.values, result.expectation_values)

Parameterized observable results:
{angle: 0.0} {'Z': 1.0, 'X': 0.0}
{angle: 1.5707963267948966} {'Z': 2.220446049250313e-16, 'X': 1.0}


### Retrieve state vectors

Each result keeps the size of its own circuit, even when one binding contains
circuits with different numbers of qubits.

In [11]:
binding = CircuitBinding(
    [
        QCircuit([X(0)], label="one qubit"),
        QCircuit([H(0), CNOT(0, 1)], label="two qubits"),
    ]
)

batch_result = run(
    binding,
    QUANTINUUMDevice.TKET_AER_STATEVECTOR_SIMULATOR,
)
print("State-vector results:")
for result in batch_result.results:
    print(result.job.circuit.label, result.job.circuit.nb_qubits, result.amplitudes)

State-vector results:
one qubit 1 [0.+0.j 1.+0.j]
two qubits 2 [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


### Run bindings remotely with Nexus

Remote Quantinuum bindings use the corresponding Nexus devices and require a
configured Nexus account and project. `submit` returns both the Nexus job ID
and the MPQP `Job`. Keep the returned `Job`, because MPQP needs it to rebuild
the individual binding results in their original order:

```python
from mpqp import get_remote_result, submit

binding = CircuitBinding(
    [QCircuit(1), QCircuit([X(0)])],
    measurements=BasisMeasure(shots=10),
)

job_id, job = submit(
    binding,
    QUANTINUUMDevice.NEXUS_AER_SIMULATOR,
)
result = get_remote_result(job)
```

The Nexus job ID identifies the remote execution, but it is not enough by
itself to reconstruct a `CircuitBinding` result.